# Baseline — TF-IDF + LogisticRegression

Baseline klasik untuk sentiment analysis. Hasil (validation): accuracy 70.2%, F1-macro 0.674.

Lihat `src/baseline.py` (versi dasar) dan `src/baseline_tuned.py` (sweep hyperparameter).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.preprocess import (load_dataset_local, apply_mapping,
                            stratified_subsample, get_texts_labels, ID2LABEL)
from src.baseline import (strip_html, preprocess_texts, build_pipeline,
                          f1_per_class, save_confusion_matrix, RESULTS_DIR)
from sklearn.metrics import accuracy_score, f1_score
print('Imports OK')

In [ ]:
# Load + mapping + subsample
ds = load_from_disk('../data/amazon_reviews_multi_en') if False else load_dataset_local()
train_full = apply_mapping(ds['train'])
val_full = apply_mapping(ds['validation'])
train_ds = stratified_subsample(train_full, 20000, seed=42)
print(f'train={len(train_ds)}  val={len(val_full)}')

In [ ]:
# Train + eval (validation)
train_texts, train_labels = get_texts_labels(train_ds)
val_texts, val_labels = get_texts_labels(val_full)
train_texts = preprocess_texts(train_texts)
val_texts = preprocess_texts(val_texts)

vectorizer, classifier = build_pipeline()
x_train = vectorizer.fit_transform(train_texts)
x_val = vectorizer.transform(val_texts)
classifier.fit(x_train, train_labels)
val_pred = classifier.predict(x_val).tolist()

print(f'accuracy = {accuracy_score(val_labels, val_pred):.4f}')
print(f'f1_macro = {f1_score(val_labels, val_pred, average="macro"):.4f}')
print('f1 per kelas:', f1_per_class(val_labels, val_pred))

In [ ]:
# Confusion matrix
from pathlib import Path
save_confusion_matrix(val_labels, val_pred, RESULTS_DIR / 'confusion_matrix_baseline.png')
import matplotlib.pyplot as plt, matplotlib.image as mpimg
img = mpimg.imread(RESULTS_DIR / 'confusion_matrix_baseline.png')
plt.figure(figsize=(6,5)); plt.imshow(img); plt.axis('off'); plt.show()